In [ ]:
# 🎭 Intelligent Excuse Generator
### AI-Powered Context-Aware Excuse Generation System
---
**Tech Stack:** Python · spaCy · NLTK · scikit-learn · ipywidgets  
**Features:** Context-aware NLP · Rule-based engine · ML believability scoring · Emergency mode · Multi-format output · Proof suggestions  
**Author:** Divyanshu | B.Tech CSE | My Capstone Project

SyntaxError: invalid character '·' (U+00B7) (2727487813.py, line 4)

In [ ]:
# ── Setup: add project root to path ─────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(""))

# ── Imports ──────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from src.nlp_engine      import process_input
from src.excuse_engine   import generate_excuse, generate_emergency_excuse
from src.scorer          import get_model, score_excuse
from src.formatter       import format_all
from src.proof_generator import get_proof_suggestions

# Pre-load ML model
print("Loading ML model...")
MODEL = get_model()
print("✅ System ready!")

Loading ML model...
✅ System ready!


In [ ]:
# ── Input Widgets ────────────────────────────────────────────

situation_input = widgets.Textarea(
    placeholder = "Describe your situation here... e.g. I missed the assignment deadline because of a power cut",
    layout      = widgets.Layout(width="100%", height="80px")
)

category_dropdown = widgets.Dropdown(
    options     = ["academic", "professional", "social", "emergency"],
    value       = "academic",
    description = "Category:",
    style       = {"description_width": "90px"}
)

relationship_dropdown = widgets.Dropdown(
    options     = ["teacher", "boss", "parent", "friend", "colleague", "client", "partner"],
    value       = "teacher",
    description = "Recipient:",
    style       = {"description_width": "90px"}
)

tone_dropdown = widgets.Dropdown(
    options     = ["formal", "casual"],
    value       = "formal",
    description = "Tone:",
    style       = {"description_width": "90px"}
)

format_toggle = widgets.ToggleButtons(
    options     = ["Plain", "Formal Letter", "WhatsApp"],
    value       = "Plain",
    description = "Format:",
    style       = {"description_width": "90px", "button_width": "120px"}
)

proof_count = widgets.IntSlider(
    value       = 3,
    min         = 1,
    max         = 5,
    description = "Proof tips:",
    style       = {"description_width": "90px"}
)

generate_btn   = widgets.Button(
    description  = "✨ Generate Excuse",
    button_style = "primary",
    layout       = widgets.Layout(width="200px", height="40px")
)

emergency_btn  = widgets.Button(
    description  = "🚨 Emergency Mode",
    button_style = "danger",
    layout       = widgets.Layout(width="200px", height="40px")
)

output_area = widgets.Output()

# ── Layout ───────────────────────────────────────────────────
display(HTML("<h3 style='margin-bottom:8px'>⚙️ Configuration</h3>"))
display(widgets.VBox([
    widgets.HTML("<b>Describe your situation:</b>"),
    situation_input,
    widgets.HBox([category_dropdown, relationship_dropdown]),
    widgets.HBox([tone_dropdown, format_toggle]),
    proof_count,
    widgets.HBox([generate_btn, emergency_btn]),
]))
display(output_area)

Output()

In [ ]:
# ── Core Display Function ────────────────────────────────────

def display_results(excuse_result, is_emergency=False):
    """Render the full output — excuse, score, proof suggestions."""
    with output_area:
        clear_output(wait=True)

        raw_excuse   = excuse_result["raw_excuse"]
        category     = excuse_result["category"]
        relationship = excuse_result["relationship"]
        event        = excuse_result["event_used"]

        # Format selection
        fmt_map  = {"Plain": "plain", "Formal Letter": "letter", "WhatsApp": "whatsapp"}
        fmt_key  = fmt_map[format_toggle.value]
        formats  = format_all(raw_excuse, relationship, tone_dropdown.value)
        output   = formats[fmt_key]

        # Believability score
        score_result = score_excuse(raw_excuse, MODEL)

        # Proof suggestions
        proof = get_proof_suggestions(category, event, count=proof_count.value)

        # Score color
        score_color = {"High": "green", "Medium": "orange", "Low": "red"}
        color = score_color[score_result["label"]]

        # ── Render ───────────────────────────────────────────
        tag = "🚨 EMERGENCY" if is_emergency else f"📂 {category.upper()}"

        display(HTML(f"""
        <div style='margin-top:20px;padding:16px;border:1px solid #ddd;
                    border-radius:10px;background:#fafafa;font-family:monospace'>
            <h3 style='margin:0 0 12px 0'>🎭 Generated Excuse &nbsp;
                <span style='font-size:13px;background:#eee;padding:3px 8px;
                             border-radius:4px'>{tag}</span>
            </h3>
            <div style='white-space:pre-wrap;background:white;padding:14px;
                        border-radius:8px;border:1px solid #e0e0e0;
                        font-size:14px;line-height:1.7'>
{output}
            </div>

            <h4 style='margin:16px 0 6px 0'>📊 Believability Score</h4>
            <div style='display:flex;align-items:center;gap:12px'>
                <span style='font-size:28px;font-weight:bold;color:{color}'>
                    {score_result["score"]}/10
                </span>
                <span style='background:{color};color:white;padding:3px 10px;
                             border-radius:12px;font-size:13px'>
                    {score_result["label"]}
                </span>
                <span style='color:#555;font-size:13px'>
                    {score_result["feedback"]}
                </span>
            </div>

            <h4 style='margin:16px 0 6px 0'>🗂️ Proof Suggestions
                <span style='font-size:12px;color:#888'>
                    (event: {event})
                </span>
            </h4>
            <ul style='margin:0;padding-left:20px;line-height:2'>
                {"".join(f"<li>{s}</li>" for s in proof["suggestions"])}
            </ul>
            <p style='margin:10px 0 0;font-size:11px;color:#e67e22'>
                {proof["disclaimer"]}
            </p>
        </div>
        """))


# ── Button Handlers ──────────────────────────────────────────

def on_generate(b):
    situation = situation_input.value.strip()
    if not situation:
        with output_area:
            clear_output(wait=True)
            display(HTML("<p style='color:red'>⚠️ Please describe your situation first.</p>"))
        return
    ctx    = process_input(situation, category_dropdown.value)
    result = generate_excuse(
        situation    = situation,
        category     = category_dropdown.value,
        relationship = relationship_dropdown.value,
        nlp_context  = ctx
    )
    display_results(result, is_emergency=False)


def on_emergency(b):
    situation = situation_input.value.strip()
    if not situation:
        situation = "I have an urgent emergency and cannot fulfill my obligations right now."
    ctx    = process_input(situation, "emergency")
    result = generate_emergency_excuse(situation, ctx)
    display_results(result, is_emergency=True)


generate_btn.on_click(on_generate)
emergency_btn.on_click(on_emergency)

print("✅ Handlers registered. Use the widgets above to generate excuses!")

✅ Handlers registered. Use the widgets above to generate excuses!
